In [1]:
import logging
import argparse

import torch

from lib.config import Config
from lib.runner import Runner
from lib.experiment import Experiment
import sys

import time
import os
import sys

import torch
# from model.lanenet.train_lanenet import train_model
# from dataloader.data_loaders import TusimpleSet
# from dataloader.transformers import Rescale
# from model.lanenet.LaneNet import LaneNet

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch.utils.data import DataLoader, Dataset

from torch.autograd import Variable
from PIL import Image
from torchvision import transforms
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    A = None
    ToTensorV2 = None

# from model.utils.cli_helper import parse_args
# from model.eval_function import Eval_Score
import sys
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt


DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(DEVICE)
from scipy.ndimage import binary_dilation, label
from matplotlib.patches import Patch
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import random
import json
from PIL import Image

from lib.config import Config

import warnings
warnings.filterwarnings('ignore')

/Users/amannindra/miniconda3/envs/Lannet310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cpu


In [2]:
def get_files(segment_folder):
    return [path for path in segment_folder.iterdir() if path.is_file()]
def get_directories(cipo_path):
    return [path for path in cipo_path.iterdir() if path.is_dir()]
def remove_first_parent(path: Path) -> Path:
    return Path(*path.parts[1:])
def check_dir(check):
    if check.is_dir():
        print(f"Directory Found: {check}")
    else:
        print(f"Directory Not Found: {check}")
def check_file(check):
    if check.is_file():
        print(f"File Found: {check}")
    else:
        print("File Not Found")


model_path = Path("/Users/amannindra/Projects/Auto/Autonomous-Bicycle/LaneATT/experiments/Testing/models/model_0020.pt")

#check_file(model_path)

In [3]:
def lanes_to_px(lanes, w, h):
    out = []
    for lane in lanes:
        pts = lane.points.copy().astype(float)
        pts[:, 0] *= w          # Lane.points are normalized (x, y) in [0, 1]
        pts[:, 1] *= h
        out.append(pts.round().astype(int))
    return out

In [9]:
config_path = "/Users/amannindra/Projects/Auto/Autonomous-Bicycle/LaneATT/experiments/Testing_Pinnacle/config.yaml"

cfg = Config(config_path)
exp = Experiment(exp_name = "Testing_Pinnacle", args = None, mode="eval")
device = torch.device('cpu')
model = cfg.get_model()
# print(model)
# runner = Runner(cfg, exp, device, view=True, resume=False, deterministic=False)
# runner.eval(epoch = 7)

print(type(model))

<class 'lib.models.laneatt.LaneATT'>


In [ ]:
infer_params = dict(cfg.get_test_parameters())# {'conf_threshold':0.5,'nms_thres':50.,'nms_topk':4}
infer_params['conf_threshold'] = None
test_dataset = cfg.get_dataset('test2')
print(infer_params)


n_show = 6
plt.figure(figsize=(18, 8))
with torch.no_grad():
    for idx in range(n_show):
        img_t, label, _ = test_dataset[idx]
        x = img_t.unsqueeze(0).to(device)

        output = model(x, **infer_params)
        pred_lanes = model.decode(output, as_lanes=True)[0]   # predicted Lane objects
        # gt_lanes   = test_dataset.label_to_lanes(label)        # ground-truth Lane objects

        # n_props = output[0][0].shape[0]                        # proposals surviving NMS
        # print(f"idx {idx}: {n_props} proposals after NMS -> {len(pred_lanes)} decoded lanes")

        # # build a displayable RGB image from the model's own input tensor (normalize:false in cfg)
        # img = (img_t.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()       # cv2.imread loads BGR

        # for pts in lanes_to_px(gt_lanes,   img.shape[1], img.shape[0]):   # GT = blue
        #     for p0, p1 in zip(pts[:-1], pts[1:]):
        #         cv2.line(img, tuple(p0), tuple(p1), (0, 0, 255), 2)
        # for pts in lanes_to_px(pred_lanes, img.shape[1], img.shape[0]):   # pred = green
        #     for p0, p1 in zip(pts[:-1], pts[1:]):
        #         cv2.line(img, tuple(p0), tuple(p1), (0, 255, 0), 3)

        # ax = plt.subplot(2, 3, idx + 1)
        # ax.imshow(img)
        # ax.set_title(f"idx {idx}: {len(pred_lanes)} pred (green) / {len(gt_lanes)} GT (blue)")
        # ax.axis('off')
plt.tight_layout(); plt.show()

[2026-06-30 10:31:43,389] [lib.datasets.culane] [INFO] Indexing CULane test annotations (lazy loading)...
[2026-06-30 10:31:43,437] [lib.datasets.culane] [INFO] 34680 annotations indexed.


{'conf_threshold': None, 'nms_thres': 50.0, 'nms_topk': 4}


<Figure size 1800x800 with 0 Axes>